# UCS420: Cognitive Computing — Assignment 4
## A Cognitive FAQ System Using Pandas (Nova 2.0)

Aadish Jain

Roll No: 1024170413

Batch: 3Q33

## Q1: Build Your Personalized Knowledge Base

In [ ]:
import pandas as pd


roll_number = "1024170413"

last_two_digits = [int(d) for d in roll_number[-2:]]

fixed_entries = [
    {"question": "what is the annual fee",
     "answer": "The annual fee is Rs 500.",
     "keywords": "fee cost price charge",
     "category": "billing"},
    {"question": "how to reset password",
     "answer": "Go to Settings > Reset Password.",
     "keywords": "password reset login",
     "category": "account"},
    {"question": "what are your working hours",
     "answer": "We are open 9 AM to 5 PM.",
     "keywords": "hours timing open time",
     "category": "general"},
    {"question": "how can i pay the fee",
     "answer": "You can pay via UPI, card, or net banking.",
     "keywords": "pay payment upi fee",
     "category": "billing"}
]

category_map = ["billing", "account", "general"]

personalized_entries = []


for d in last_two_digits:
    category = category_map[d % 3]

    if category == "billing":
        entry = {
            "question": "how can i check my latest payment status",
            "answer": "You can check your latest payment status from the Billing section.",
            "keywords": "payment status billing transaction",
            "category": category
        }
    elif category == "account":
        entry = {
            "question": "how do i update my registered mobile number",
            "answer": "Go to Account Settings and update your registered mobile number.",
            "keywords": "mobile number update account",
            "category": category
        }
    else:
        entry = {
            "question": "where can i find general help",
            "answer": "You can find general help in the Help and Support section.",
            "keywords": "help support information general",
            "category": category
        }

    personalized_entries.append(entry)

df = pd.DataFrame(fixed_entries + personalized_entries)

print("Roll Number:", roll_number)
print("\nFinal 6-row Knowledge Base:")
display(df)

## Q2: Generate and Score a Hypothesis

In [ ]:
def score_query(query, df):
    """
    Score FAQ entries based on how many query words
    match the entry's question or keywords.
    Returns all entries ranked by confidence score.
    """
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        question_words = set(row["question"].lower().split())
        keyword_words = set(row["keywords"].lower().split())

        matched_words = query_words & (question_words | keyword_words)
        score = len(matched_words)

        results.append({
            "index": index,
            "question": row["question"],
            "answer": row["answer"],
            "category": row["category"],
            "matched_words": ", ".join(sorted(matched_words)),
            "score": score
        })

    return pd.DataFrame(results).sort_values(
        by="score", ascending=False
    ).reset_index(drop=True)

query = input("Enter your FAQ query: ")
ranked_results = score_query(query, df)

print("\nRanked Results:")
display(ranked_results)

## Q3: Find FAQs from the Same Category

In [ ]:
def same_category(category_name, df):
    return df[df["category"].str.lower() == category_name.lower()]


selected_category = personalized_entries[0]["category"]

print("Selected category:", selected_category)
print("\nFAQs belonging to this category:")
display(same_category(selected_category, df))

## Q4: Add a New Keyword and Save the DataFrame

In [ ]:

entry_index = 0

print("Selected entry:")
display(df.loc[[entry_index]])

new_keyword = input("Enter a new keyword to add: ").strip().lower()

if new_keyword:
    existing_keywords = df.at[entry_index, "keywords"].split()
    if new_keyword not in existing_keywords:
        df.at[entry_index, "keywords"] += " " + new_keyword

csv_filename = f"{roll_number}_faq_data.csv"
df.to_csv(csv_filename, index=False)

print(f"\nUpdated DataFrame saved as: {csv_filename}")
display(df)

## Q5: Count FAQ Entries per Category

In [ ]:
category_counts = df.groupby("category").size()

print("Number of FAQ entries per category:")
print(category_counts)

## Q6: Modified Scoring Function with Tie Handling

In [ ]:
def score_query_with_ties(query, df):
    """
    Scores every FAQ entry and prints all entries tied
    for the highest score.
    """
    query_words = set(query.lower().split())
    results = []

    for index, row in df.iterrows():
        question_words = set(row["question"].lower().split())
        keyword_words = set(row["keywords"].lower().split())

        matched_words = query_words & (question_words | keyword_words)
        score = len(matched_words)

        results.append({
            "index": index,
            "question": row["question"],
            "answer": row["answer"],
            "category": row["category"],
            "matched_words": ", ".join(sorted(matched_words)),
            "score": score
        })

    results_df = pd.DataFrame(results).sort_values(
        by="score", ascending=False
    ).reset_index(drop=True)

    highest_score = results_df["score"].max()
    best_matches = results_df[results_df["score"] == highest_score]

    print(f"Highest confidence score: {highest_score}")

    if len(best_matches) > 1:
        print("\nTie detected! All equally matching entries:")
        display(best_matches)
    else:
        print("\nBest matching entry:")
        display(best_matches)

    return results_df


print("===== TIE EXAMPLE =====")
tie_results = score_query_with_ties("fee", df)

print("\n===== NON-TIE EXAMPLE =====")
non_tie_results = score_query_with_ties("password reset login", df)